In [1]:
import torch
import torch.nn.functional as F
import math

In [2]:
# ---------------------------------------------------------------------------
# Core Function
# ---------------------------------------------------------------------------
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled dot-product attention as defined in 'Attention Is All You Need'.

    Args:
        Q   : Query  tensor  (..., T_q, d_k)
        K   : Key    tensor  (..., T_k, d_k)
        V   : Value  tensor  (..., T_k, d_v)
        mask: optional bool tensor (..., T_q, T_k) -- True = mask out

    Returns:
        output       : (..., T_q, d_v)  weighted sum of values
        weights      : (..., T_q, T_k)  attention probabilities
        raw_scores   : (..., T_q, T_k)  unscaled QK^T
        scaled_scores: (..., T_q, T_k)  QK^T / sqrt(d_k)
    """
    d_k = Q.size(-1)

    # Step 1: raw dot products  QK^T
    raw_scores = torch.matmul(Q, K.transpose(-2, -1))        # (..., T_q, T_k)

    # Step 2: scale by 1/sqrt(d_k)
    scaled_scores = raw_scores / math.sqrt(d_k)

    # Step 3: optional masking (set masked positions to -inf before softmax)
    if mask is not None:
        scaled_scores = scaled_scores.masked_fill(mask, float("-inf"))

    # Step 4: softmax -> attention weights
    weights = F.softmax(scaled_scores, dim=-1)

    # Step 5: weighted sum of values
    output = torch.matmul(weights, V)

    return output, weights, raw_scores, scaled_scores



In [3]:
# ---------------------------------------------------------------------------
# Softmax Stability Check
# ---------------------------------------------------------------------------
def softmax_stability_check(raw_scores, scaled_scores):
    """
    Compare softmax behavior on raw (unscaled) vs. scaled scores.
    Large dot-product magnitudes push softmax toward one-hot,
    producing near-zero gradients (saturation).
    Dividing by sqrt(d_k) restores variance ~1, keeping gradients healthy.
    """
    d_k = scaled_scores.shape[-1]
    print("\n" + "=" * 60)
    print("Softmax Stability Check")
    print("=" * 60)

    # --- Before scaling ---
    sm_raw = F.softmax(raw_scores, dim=-1)
    print(f"\n[BEFORE scaling]  (raw scores, d_k={d_k})")
    print(f"  score mean : {raw_scores.mean().item():.4f}")
    print(f"  score std  : {raw_scores.std().item():.4f}")
    print(f"  score max  : {raw_scores.max().item():.4f}")
    print(f"  score min  : {raw_scores.min().item():.4f}")
    print(f"  softmax avg-max-prob  : {sm_raw.max(dim=-1).values.mean().item():.4f}"
          f"  (near 1.0 = peaked/saturated)")
    print(f"  softmax avg-min-prob  : {sm_raw.min(dim=-1).values.mean().item():.6f}")
    print(f"  entropy (avg)         : {(-sm_raw * (sm_raw + 1e-9).log()).sum(-1).mean().item():.4f}")

    # --- After scaling ---
    sm_scaled = F.softmax(scaled_scores, dim=-1)
    print(f"\n[AFTER  scaling]  (divided by sqrt({d_k}) = {math.sqrt(d_k):.2f})")
    print(f"  score mean : {scaled_scores.mean().item():.4f}")
    print(f"  score std  : {scaled_scores.std().item():.4f}")
    print(f"  score max  : {scaled_scores.max().item():.4f}")
    print(f"  score min  : {scaled_scores.min().item():.4f}")
    print(f"  softmax avg-max-prob  : {sm_scaled.max(dim=-1).values.mean().item():.4f}"
          f"  (closer to uniform = more gradient)")
    print(f"  softmax avg-min-prob  : {sm_scaled.min(dim=-1).values.mean().item():.6f}")
    print(f"  entropy (avg)         : {(-sm_scaled * (sm_scaled + 1e-9).log()).sum(-1).mean().item():.4f}")

    print("\nConclusion:")
    print(f"  For d_k={d_k} unit-variance Q,K, dot products have std ~ sqrt({d_k})={math.sqrt(d_k):.1f}.")
    print(f"  Without scaling, softmax saturates (high max-prob, near-zero gradients).")
    print(f"  Dividing by sqrt(d_k) restores std~1 -> balanced weights -> healthy gradients.")


In [4]:
# ---------------------------------------------------------------------------
# Tests
# ---------------------------------------------------------------------------
def main():
    torch.manual_seed(0)

    # -----------------------------------------------------------------------
    # Test 1: Small hand-checkable example
    # -----------------------------------------------------------------------
    print("=" * 60)
    print("Test 1 - Small Deterministic Example  (T=4, d_k=4)")
    print("=" * 60)
    T, d_k, d_v = 4, 4, 4
    Q = torch.tensor([[1., 0., 1., 0.],
                       [0., 1., 0., 1.],
                       [1., 1., 0., 0.],
                       [0., 0., 1., 1.]])
    K = torch.tensor([[1., 0., 0., 1.],
                       [0., 1., 1., 0.],
                       [1., 0., 1., 0.],
                       [0., 1., 0., 1.]])
    V = torch.tensor([[ 1.,  2.,  3.,  4.],
                       [ 5.,  6.,  7.,  8.],
                       [ 9., 10., 11., 12.],
                       [13., 14., 15., 16.]])

    output, weights, raw, scaled = scaled_dot_product_attention(Q, K, V)

    print(f"\nQ  ({T}x{d_k}):\n{Q}")
    print(f"\nK  ({T}x{d_k}):\n{K}")
    print(f"\nV  ({T}x{d_v}):\n{V}")
    print(f"\nRaw scores  QK^T ({T}x{T}):\n{raw.round(decimals=3)}")
    print(f"\nScaled scores  QK^T/sqrt({d_k}) ({T}x{T}):\n{scaled.round(decimals=3)}")
    print(f"\nAttention weight matrix  softmax(scaled) ({T}x{T}):")
    print(weights.round(decimals=4))
    print(f"\nRow sums (should all be 1.0): {weights.sum(dim=-1).tolist()}")
    print(f"\nOutput vectors ({T}x{d_v}):")
    print(output.round(decimals=4))

    # -----------------------------------------------------------------------
    # Test 2: Random inputs + stability check
    # -----------------------------------------------------------------------
    print("\n" + "=" * 60)
    print("Test 2 - Random Inputs  (T=8, d_k=64)")
    print("=" * 60)
    T2, d_k2 = 8, 64
    Q2 = torch.randn(T2, d_k2)
    K2 = torch.randn(T2, d_k2)
    V2 = torch.randn(T2, d_k2)

    output2, weights2, raw2, scaled2 = scaled_dot_product_attention(Q2, K2, V2)
    print(f"\nInputs: Q,K,V each ({T2}x{d_k2}), sampled from N(0,1)")
    print(f"Output shape : {output2.shape}")
    print(f"\nAttention weight matrix (all {T2}x{T2}):")
    print(weights2.round(decimals=4))
    print(f"\nOutput vectors (first 4 rows, first 8 cols):")
    print(output2[:4, :8].round(decimals=4))

    # Stability check (add batch dim for consistent interface)
    softmax_stability_check(raw2.unsqueeze(0), scaled2.unsqueeze(0))

    # -----------------------------------------------------------------------
    # Test 3: Batched multi-head shape
    # -----------------------------------------------------------------------
    print("\n" + "=" * 60)
    print("Test 3 - Batched + Multi-Head  (B=2, H=4, T=6, d_k=16)")
    print("=" * 60)
    B, H, T3, d_k3 = 2, 4, 6, 16
    Q3 = torch.randn(B, H, T3, d_k3)
    K3 = torch.randn(B, H, T3, d_k3)
    V3 = torch.randn(B, H, T3, d_k3)
    out3, w3, _, _ = scaled_dot_product_attention(Q3, K3, V3)
    print(f"Output shape  : {out3.shape}   (expected {B},{H},{T3},{d_k3})")
    print(f"Weights shape : {w3.shape}    (expected {B},{H},{T3},{T3})")
    assert w3.shape == (B, H, T3, T3)
    assert torch.allclose(w3.sum(dim=-1), torch.ones(B, H, T3), atol=1e-5)
    print("Shape and row-sum assertions passed.")

    # -----------------------------------------------------------------------
    # Test 4: Causal (look-ahead) mask
    # -----------------------------------------------------------------------
    print("\n" + "=" * 60)
    print("Test 4 - Causal Mask  (T=5, d_k=8)")
    print("=" * 60)
    T4, d_k4 = 5, 8
    Q4 = torch.randn(T4, d_k4)
    K4 = torch.randn(T4, d_k4)
    V4 = torch.randn(T4, d_k4)
    causal = torch.triu(torch.ones(T4, T4, dtype=torch.bool), diagonal=1)
    out4, w4, _, _ = scaled_dot_product_attention(Q4, K4, V4, mask=causal)
    print(f"\nCausal mask (True = blocked):\n{causal.int()}")
    print(f"\nMasked attention weights (upper triangle -> 0):")
    print(w4.round(decimals=4))
    blocked_sum = w4[causal].sum().item()
    print(f"\nSum of blocked positions: {blocked_sum:.6f}  (should be ~0)")
    assert blocked_sum < 1e-5, "Masking failed!"
    print("Causal masking assertion passed.")


if __name__ == "__main__":
    main()


Test 1 - Small Deterministic Example  (T=4, d_k=4)

Q  (4x4):
tensor([[1., 0., 1., 0.],
        [0., 1., 0., 1.],
        [1., 1., 0., 0.],
        [0., 0., 1., 1.]])

K  (4x4):
tensor([[1., 0., 0., 1.],
        [0., 1., 1., 0.],
        [1., 0., 1., 0.],
        [0., 1., 0., 1.]])

V  (4x4):
tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.],
        [13., 14., 15., 16.]])

Raw scores  QK^T (4x4):
tensor([[1., 1., 2., 0.],
        [1., 1., 0., 2.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

Scaled scores  QK^T/sqrt(4) (4x4):
tensor([[0.5000, 0.5000, 1.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 1.0000],
        [0.5000, 0.5000, 0.5000, 0.5000],
        [0.5000, 0.5000, 0.5000, 0.5000]])

Attention weight matrix  softmax(scaled) (4x4):
tensor([[0.2350, 0.2350, 0.3875, 0.1425],
        [0.2350, 0.2350, 0.1425, 0.3875],
        [0.2500, 0.2500, 0.2500, 0.2500],
        [0.2500, 0.2500, 0.2500, 0.2500]])

Row sums (should all be 1.0): [